# Newton's Method — Fast, but Only Locally

**Continuous Optimization (MasterMath) — Lecture 9 demo.**

Newton's method minimises $f$ by repeatedly minimising its second-order Taylor model
$$x_{k+1} \in \arg\min_x\ f(x_k) + f'(x_k)(x-x_k) + \tfrac12 f''(x_k)(x-x_k)^2,$$
which gives the iteration
$$x_{k+1} = x_k - f''(x_k)^{-1} f'(x_k).$$

We run it on the two examples from the slides:

- **Example I:** $f(x) = 7x - \log x$ on $x>0$, with minimiser $x^\star = 1/7$, $f'(x) = 7 - 1/x$ and $f''(x) = 1/x^2$. The Newton step simplifies to $x_{k+1} = 2x_k - 7x_k^2$.
- **Example II:** $f(x) = \log(e^{2x} + e^{-2x})$, with minimiser $x^\star = 0$, $f'(x) = 2\tanh(2x)$ and $f''(x) = 4/\cosh^2(2x)$. The Newton step simplifies to $x_{k+1} = x_k - \tfrac14\sinh(4x_k)$.

Close to $x^\star$ the number of correct digits roughly **doubles** in every step (quadratic convergence). Far from $x^\star$ the method can fail.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

EXAMPLES = {
    "I": dict(
        name="Example I: f(x) = 7x - log(x)",
        f=lambda x: 7 * x - np.log(x),
        fp=lambda x: 7 - 1 / x,
        fpp=lambda x: 1 / x**2,
        domain=lambda x: x > 0,
        xstar=1 / 7,
        plot_range=(1e-3, 0.5),
    ),
    "II": dict(
        name="Example II: f(x) = log(exp(2x) + exp(-2x))",
        f=lambda x: np.logaddexp(2 * x, -2 * x),
        fp=lambda x: 2 * np.tanh(2 * x),
        fpp=lambda x: 4 / np.cosh(2 * x) ** 2,
        domain=lambda x: np.isfinite(x) and abs(x) < 1e3,
        xstar=0.0,
        plot_range=(-2.0, 2.0),
    ),
}


def newton(ex, x0, N):
    """Run N Newton steps from x0. Stops early if an iterate leaves the domain."""
    xs = [x0]
    status = "converging"
    for _ in range(N):
        x = xs[-1]
        with np.errstate(over="ignore", divide="ignore"):
            x_new = x - ex["fp"](x) / ex["fpp"](x)
        xs.append(x_new)
        if not ex["domain"](x_new):
            status = "left the domain" if np.isfinite(x_new) else "diverged"
            break
        if abs(x_new - ex["xstar"]) < 1e-16:
            break
    return np.array(xs), status


def show(key, x0, N=8):
    """Print the iterates and plot f with the Newton steps and the error |x_k - x*|."""
    ex = EXAMPLES[key]
    xs, status = newton(ex, x0, N)

    print(ex["name"], f"  x0 = {x0}")
    print(" k            x_k        |x_k - x*|")
    for k, x in enumerate(xs):
        print(f"{k:2d}  {x:16.10g}  {abs(x - ex['xstar']):12.3e}")
    print("Status:", status)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    # Left: f together with the quadratic models used by Newton's method.
    lo, hi = ex["plot_range"]
    t = np.linspace(lo, hi, 1000)
    ax1.plot(t, ex["f"](t), "k", label="$f$")
    for k, x in enumerate(xs[:-1]):
        if not (lo <= x <= hi):
            break
        q = ex["f"](x) + ex["fp"](x) * (t - x) + 0.5 * ex["fpp"](x) * (t - x) ** 2
        ax1.plot(t, q, color="tab:red", alpha=0.5, lw=1,
                 label="quadratic model" if k == 0 else None)
        ax1.plot(x, ex["f"](x), "o", color="tab:red", ms=4)
        ax1.annotate(f"$x_{k}$", (x, ex["f"](x)), textcoords="offset points",
                     xytext=(4, 4), fontsize=8)
    ax1.axvline(ex["xstar"], color="gray", ls=":", lw=1)
    fmin, fmax = ex["f"](t).min(), ex["f"](t).max()
    ax1.set_ylim(fmin - 0.1 * (fmax - fmin), fmax)
    ax1.set_xlim(lo, hi)
    ax1.set_xlabel("$x$")
    ax1.set_title(ex["name"])
    ax1.legend(loc="upper center")

    # Right: the error on a log scale (quadratic convergence = ever steeper drop).
    err = np.abs(xs - ex["xstar"])
    k = np.arange(len(xs))
    ax2.semilogy(k[err > 0], err[err > 0], "o-", color="tab:red")
    ax2.set_xlabel("$k$")
    ax2.set_ylabel("$|x_k - x^\\star|$")
    ax2.set_title(f"Status: {status}")
    ax2.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

## Interactive demo

Choose an example and a starting point $x_0$. The left plot shows $f$ (black) and the quadratic models (red) that Newton's method minimises in every step. The right plot shows the error $|x_k - x^\star|$ on a log scale: quadratic convergence shows up as an ever steeper drop.

In [ ]:
import ipywidgets as widgets

example = widgets.Dropdown(options=[("Example I: 7x - log(x)", "I"),
                                    ("Example II: log(exp(2x) + exp(-2x))", "II")],
                           value="I", description="Example")
x0 = widgets.FloatSlider(value=0.01, min=0.005, max=0.5, step=0.005,
                         description="$x_0$", readout_format=".3f")
N = widgets.IntSlider(value=8, min=1, max=15, description="steps $N$")

def update_range(change):
    # Example I lives on x > 0; Example II on the whole real line.
    if change["new"] == "I":
        x0.min, x0.max, x0.step, x0.value = 0.005, 0.5, 0.005, 0.01
    else:
        x0.min, x0.max, x0.step, x0.value = -1.0, 1.0, 0.01, 0.5

example.observe(update_range, names="value")

out = widgets.interactive_output(lambda example, x0, N: show(example, x0, N),
                                 dict(example=example, x0=x0, N=N))
display(widgets.VBox([widgets.HBox([example, x0, N]), out]))

## Things to try

1. **Example I, $x_0 = 1$:** the first step lands at $x_1 = -5$, outside the domain of $f$ (Failure I on the slides).
2. **Example I, $x_0 = 0.01$ versus $x_0 = 0.1$:** from far away progress is slow at first; once close, the error drops from $10^{-5}$ to $10^{-9}$ in one step.
   Since $x_{k+1} - \tfrac17 = -7\,(x_k - \tfrac17)^2$, Newton's method converges exactly when $x_0 \in (0, \tfrac27)$. Check this with the slider.
3. **Example II, $x_0 = 0.5$ versus $x_0 = 0.6$:** the first converges, the second oscillates with growing amplitude and diverges (Failure II on the slides), although $f$ is convex and smooth everywhere.
   Find (approximately) the threshold for $|x_0|$ at which divergence starts.
4. **Quadratic convergence:** in the tables, count how many digits of $x_k$ are correct in each step once the iterates are close to $x^\star$.